In [24]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

import networkx as nx
import folium
import osmnx as ox

DEF_START = "1600 Pennsylvania Ave NW, Washington, DC 20500" # White House
DEF_END = "704 26th St NE, Washington, DC 20002" # Phelps High School

In [48]:
def geocode_location(prompt_text):
    while True:
        location = input(prompt_text)
        try:
            geo = ox.geocode(location)
            return geo, location
        except Exception:
            print("Location not found. Please query again.")

def loc_prompt():
    """
    :return: start and end geo locations
    """
    geo_start, loc_start = geocode_location("Please enter your starting location: ")
    geo_end, loc_end = geocode_location("Please enter your destination: ")

    print(f"Start Point: {geo_start} \'{loc_start}\'")
    print(f"End Point: {geo_end} \'{loc_end}\'")

    list = [geo_start, geo_end]
    return list

def center(list):
    """
    :param start: start address
    :param end: end address
    :return: center point and the correct radius for a plot
    """
    start_lat = list[0][0]
    start_lon = list[0][1]
    end_lat = list[1][0]
    end_lon = list[1][1]

    geo_center = (
        ( float(start_lat)+float(end_lat) ) / 2,
        ( float(start_lon)+float(end_lon) ) / 2
                 )

    # Euclidean distance between locations * 1.25 for some bezels for mapping
    bezel_factor = 1.25
    geo_dist = ox.distance.great_circle(
        float(list[0][0]), float(list[1][0]),
        float(list[1][0]), float(list[1][1])
    ) * bezel_factor

    print("\n")

    return geo_center, geo_dist

In [4]:
G = ox.graph_from_place("Washington, District of Columbia, USA",        # ~ 2 minutes
                        network_type="walk")

In [6]:
# Convert to GeoDataFrames if needed
nodes, edges = ox.graph_to_gdfs(G)      # ~ 9 sec

In [10]:
# Export shapeflies ~ 30 seconds
output_dir = os.path.join(os.getcwd(), 'Network Outputs')

if os.path.exists(output_dir):
    nodes.to_file(os.path.join(output_dir, "nodes.shp"))
    edges.to_file(os.path.join(output_dir, "edges.shp"))
else:
    os.mkdir(output_dir)
    nodes.to_file(os.path.join(output_dir, "nodes.shp"))
    edges.to_file(os.path.join(output_dir, "edges.shp"))

/var/folders/mv/9l39xh61679g3fk2_5cyrrzh0000gn/T/ipykernel_62263/2327985704.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  nodes.to_file(os.path.join(output_dir, "nodes.shp"))
/opt/anaconda3/envs/DC-Metro-Crime/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'street_count' to 'street_cou'
  ogr_write(


In [50]:
geo_center, geo_dist = center(loc_prompt())

print(f"Center Point: {geo_center} ")
print(f"Graph Radius: {geo_dist:.2f} m") # check units?

Start Point: (38.8854281, -77.0414735) 'national mall'
End Point: (38.8976387, -77.0365528) 'white house'


Center Point: (38.8915334, -77.03901314999999) 
Graph Radius: 11477239.09 m
